# Hybrid Fusion — 3 Improvements

Comparing 3 ways to improve Hybrid Fusion (XGBoost + TabTransformer):
1. **Leaf indices + PCA**: Replace 10 logits with 500 leaf indices → PCA 32 → concat with Transformer embedding
2. **Joint training**: Unfreeze Transformer, train Transformer + fusion MLP end-to-end
3. **Weighted soft voting**: Learn 20 weights (10 per class per model) for soft voting

## Setup
Upload to Google Drive `Colab Notebooks/anomaly-data/`:
- `data/processed/split/*.npy` (6 files)
- `models_saved/xgboost_v1.json`
- `models_saved/tabtransformer_model.pth`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
import xgboost as xgb
import matplotlib.pyplot as plt
import time, os

DRIVE_DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/anomaly-data'
DRIVE_OUT_DIR  = '/content/drive/MyDrive/Colab Notebooks/anomaly-models'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
X_train = np.load(f'{DRIVE_DATA_DIR}/X_train.npy').astype(np.float32)
X_val   = np.load(f'{DRIVE_DATA_DIR}/X_val.npy').astype(np.float32)
X_test  = np.load(f'{DRIVE_DATA_DIR}/X_test.npy').astype(np.float32)
X_train_scale = np.load(f'{DRIVE_DATA_DIR}/X_train_scale.npy').astype(np.float32)
X_val_scale   = np.load(f'{DRIVE_DATA_DIR}/X_val_scale.npy').astype(np.float32)
X_test_scale  = np.load(f'{DRIVE_DATA_DIR}/X_test_scale.npy').astype(np.float32)
y_train = np.load(f'{DRIVE_DATA_DIR}/y_train.npy').flatten().astype(np.int64)
y_val   = np.load(f'{DRIVE_DATA_DIR}/y_val.npy').flatten().astype(np.int64)
y_test  = np.load(f'{DRIVE_DATA_DIR}/y_test.npy').flatten().astype(np.int64)

n_features, n_classes = 76, 10
class_names = ['Benign','Analysis','Backdoor','DoS','Exploits',
               'Fuzzers','Generic','Reconnaissance','Shellcode','Worms']
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
print('Loading XGBoost...')
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(f'{DRIVE_DATA_DIR}/xgboost_v1.json')

# XGBoost raw logits (for approach 2, 3)
train_xgb_logits = xgb_model.predict(X_train, output_margin=True).astype(np.float32)
val_xgb_logits   = xgb_model.predict(X_val,   output_margin=True).astype(np.float32)
test_xgb_logits  = xgb_model.predict(X_test,  output_margin=True).astype(np.float32)

# XGBoost probabilities (for approach 3)
train_xgb_prob = xgb_model.predict_proba(X_train).astype(np.float32)
val_xgb_prob   = xgb_model.predict_proba(X_val).astype(np.float32)
test_xgb_prob  = xgb_model.predict_proba(X_test).astype(np.float32)

# XGBoost leaf indices (for approach 1)
print('Extracting leaf indices...')
train_leaves = xgb_model.apply(X_train).astype(np.int64)
val_leaves   = xgb_model.apply(X_val).astype(np.int64)
test_leaves  = xgb_model.apply(X_test).astype(np.int64)
print(f'Leaf indices — train: {train_leaves.shape}, val: {val_leaves.shape}')

In [ ]:
print('Loading TabTransformer...')

D_MODEL, NHEAD, N_LAYERS, D_FF = 32, 4, 4, 128

class TabTransformerFull(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_embedding = nn.Linear(1, D_MODEL)
        self.pos_encoding = nn.Parameter(torch.randn(1, n_features, D_MODEL) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NHEAD, dim_feedforward=D_FF,
            dropout=0.2, activation='relu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS)
        self.pool = nn.Sequential(
            nn.LayerNorm(D_MODEL),
            nn.Linear(D_MODEL, D_MODEL // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(D_MODEL // 2, n_classes),
        )

    def forward(self, x, return_embed=False):
        x = x.unsqueeze(-1)
        x = self.feature_embedding(x)
        x = x + self.pos_encoding
        x = self.transformer(x)
        embed = x.mean(dim=1)  # (B, 32)
        if return_embed:
            return embed, self.pool(embed)
        return self.pool(embed)

saved = torch.load(f'{DRIVE_DATA_DIR}/tabtransformer_model.pth', map_location='cpu', weights_only=False)
tt_model = TabTransformerFull()
tt_model.load_state_dict(saved['model_state_dict'])
tt_model.to(device)
tt_model.eval()

# Extract embeddings (shared across approaches)
def extract_tt_embed(X, name):
    print(f'  TT {name}...')
    res = []
    with torch.no_grad():
        for i in range(0, len(X), 4096):
            batch = torch.from_numpy(X[i:i+4096]).to(device)
            embed, _ = tt_model(batch, return_embed=True)
            res.append(embed.cpu().numpy())
    return np.concatenate(res, axis=0).astype(np.float32)

train_tt_embed = extract_tt_embed(X_train_scale, 'train')
val_tt_embed   = extract_tt_embed(X_val_scale, 'val')
test_tt_embed  = extract_tt_embed(X_test_scale, 'test')
print(f'TT embed — train: {train_tt_embed.shape}')

## Approach 1: Leaf indices + PCA → Fusion MLP

**Why:** The original Hybrid used 10 XGBoost logits (before softmax). These contain less information than the full tree paths. Leaf indices capture exactly which leaf each of the 500 trees assigned — preserving more of XGBoost's decision logic. PCA compresses 500 column indices down to 32 dense dimensions, then we concat with TT's 32-dim embedding.

**Pipeline:** `500 leaves → PCA(32) → concat(TT embed 32) → 64 → Fusion MLP`

In [ ]:
print('Fitting PCA on leaf indices...')
pca = PCA(n_components=32, random_state=42)
train_leaf_pca = pca.fit_transform(train_leaves).astype(np.float32)
val_leaf_pca   = pca.transform(val_leaves).astype(np.float32)
test_leaf_pca  = pca.transform(test_leaves).astype(np.float32)
print(f'Leaf PCA — train: {train_leaf_pca.shape}, explained var: {pca.explained_variance_ratio_.sum():.3f}')

X_train_a1 = np.concatenate([train_leaf_pca, train_tt_embed], axis=1)
X_val_a1   = np.concatenate([val_leaf_pca,   val_tt_embed],   axis=1)
X_test_a1  = np.concatenate([test_leaf_pca,  test_tt_embed],  axis=1)
print(f'Fusion input dim: {X_train_a1.shape[1]}')

In [ ]:
class FusionMLP(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, n_classes),
        )
    def forward(self, x):
        return self.net(x)

def train_fusion(X_tr, X_v, y_tr, y_v, in_dim, lr=5e-4, epochs=100, patience=10):
    model = FusionMLP(in_dim, n_classes).to(device)
    bs = 4096
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=bs, shuffle=True)
    val_loader   = DataLoader(TensorDataset(torch.from_numpy(X_v),  torch.from_numpy(y_v)),  batch_size=bs, shuffle=False)

    class_counts = np.bincount(y_tr.astype(int))
    cw = torch.tensor(len(y_tr) / (n_classes * class_counts), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_loss, best_state, counter = float('inf'), None, 0
    for epoch in range(1, epochs + 1):
        model.train(); tr_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); loss = criterion(model(Xb), yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); tr_loss += loss.item() * Xb.size(0)
        sched.step(); tr_loss /= len(y_tr)

        model.eval(); vl_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                vl_loss += criterion(model(Xb), yb).item() * Xb.size(0)
        vl_loss /= len(y_v)

        if vl_loss < best_loss:
            best_loss = vl_loss; best_state = model.state_dict().copy(); counter = 0
        else: counter += 1
        if counter >= patience: break

    model.load_state_dict(best_state)
    return model

model_a1 = train_fusion(X_train_a1, X_val_a1, y_train, y_val, X_train_a1.shape[1])

model_a1.eval(); preds = []
with torch.no_grad():
    for i in range(0, len(X_test_a1), 4096):
        Xb = torch.from_numpy(X_test_a1[i:i+4096]).to(device)
        preds.extend(torch.argmax(model_a1(Xb), dim=1).cpu().numpy())
macro_a1 = f1_score(y_test, preds, average='macro')
wgt_a1   = f1_score(y_test, preds, average='weighted')
print(f'\nApproach 1 (Leaf PCA + TT) → Macro F1: {macro_a1:.4f}, Weighted: {wgt_a1:.4f}')
print(classification_report(y_test, preds, target_names=class_names, digits=4))

## Approach 2: Joint training (unfreeze Transformer)

**Why:** The original Hybrid froze the Transformer and only trained the fusion MLP. This meant the Transformer couldn't adapt to complement XGBoost. By unfreezing the Transformer and training both together, the Transformer can learn to focus on the patterns XGBoost misses.

**Pipeline:** `scaled features → TT (unfrozen) → embed(32) → concat(XGB logits 10) → 42 → Fusion MLP`

**Note:** Lower LR (1e-4) and smaller batch size (1024) to avoid destroying pre-trained TT weights.

In [ ]:
class JointHybrid(nn.Module):
    def __init__(self, tt_model):
        super().__init__()
        self.feature_embedding = nn.Linear(1, D_MODEL)
        self.pos_encoding = nn.Parameter(torch.randn(1, n_features, D_MODEL) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NHEAD, dim_feedforward=D_FF,
            dropout=0.2, activation='relu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS)

        # Load pre-trained weights (everything except the old TT head)
        pretrained = tt_model.state_dict()
        self.feature_embedding.weight.data.copy_(pretrained['feature_embedding.weight'])
        self.feature_embedding.bias.data.copy_(pretrained['feature_embedding.bias'])
        self.pos_encoding.data.copy_(pretrained['pos_encoding'])
        for name, param in pretrained.items():
            if 'transformer' in name and 'pool' not in name:
                self.state_dict()[name].data.copy_(param)

        # Fusion head: TT embed (32) + XGB logits (10) = 42
        self.fusion = nn.Sequential(
            nn.Linear(D_MODEL + 10, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, n_classes),
        )

    def forward(self, x_scale, xgb_logits):
        x = x_scale.unsqueeze(-1)
        x = self.feature_embedding(x)
        x = x + self.pos_encoding
        x = self.transformer(x)
        embed = x.mean(dim=1)
        combined = torch.cat([embed, xgb_logits], dim=1)
        return self.fusion(combined)

model_a2 = JointHybrid(tt_model).to(device)
total_params = sum(p.numel() for p in model_a2.parameters())
trainable = sum(p.numel() for p in model_a2.parameters() if p.requires_grad)
print(f'Params: {total_params:,} total, {trainable:,} trainable')

In [ ]:
print('\nJoint training (TT unfrozen)...')
bs = 1024
train_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_train_scale), torch.from_numpy(train_xgb_logits), torch.from_numpy(y_train)
), batch_size=bs, shuffle=True)
val_loader = DataLoader(TensorDataset(
    torch.from_numpy(X_val_scale), torch.from_numpy(val_xgb_logits), torch.from_numpy(y_val)
), batch_size=bs, shuffle=False)

class_counts = np.bincount(y_train.astype(int))
cw = torch.tensor(len(y_train) / (n_classes * class_counts), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=cw)
opt = optim.AdamW(model_a2.parameters(), lr=1e-4, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

best_loss, best_state, counter = float('inf'), None, 0
epochs, patience = 50, 8
start = time.time()

for epoch in range(1, epochs + 1):
    model_a2.train(); tr_loss = 0.0
    for Xb, xgb_b, yb in train_loader:
        Xb, xgb_b, yb = Xb.to(device), xgb_b.to(device), yb.to(device)
        opt.zero_grad(); loss = criterion(model_a2(Xb, xgb_b), yb)
        loss.backward(); nn.utils.clip_grad_norm_(model_a2.parameters(), 3.0)
        opt.step(); tr_loss += loss.item() * Xb.size(0)
    sched.step(); tr_loss /= len(y_train)

    model_a2.eval(); vl_loss = 0.0
    with torch.no_grad():
        for Xb, xgb_b, yb in val_loader:
            Xb, xgb_b, yb = Xb.to(device), xgb_b.to(device), yb.to(device)
            vl_loss += criterion(model_a2(Xb, xgb_b), yb).item() * Xb.size(0)
    vl_loss /= len(y_val)

    if vl_loss < best_loss:
        best_loss = vl_loss; best_state = model_a2.state_dict().copy(); counter = 0
    else: counter += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}/{epochs} | Train: {tr_loss:.4f} | Val: {vl_loss:.4f} | {time.time()-start:.0f}s')
    if counter >= patience:
        print(f'Early stopping at epoch {epoch}')
        break

model_a2.load_state_dict(best_state)

model_a2.eval(); preds = []
with torch.no_grad():
    for i in range(0, len(X_test_scale), bs):
        Xb = torch.from_numpy(X_test_scale[i:i+bs]).to(device)
        xgb_b = torch.from_numpy(test_xgb_logits[i:i+bs]).to(device)
        preds.extend(torch.argmax(model_a2(Xb, xgb_b), dim=1).cpu().numpy())
macro_a2 = f1_score(y_test, preds, average='macro')
wgt_a2   = f1_score(y_test, preds, average='weighted')
print(f'\nApproach 2 (Joint training) → Macro F1: {macro_a2:.4f}, Weighted: {wgt_a2:.4f}')
print(classification_report(y_test, preds, target_names=class_names, digits=4))

## Approach 3: Weighted soft voting

**Why:** The simplest possible fusion — learn just 20 scalar weights (10 per class for XGBoost + 10 for Transformer). Unlike the fusion MLP which has thousands of parameters, this is nearly impossible to overfit. It simply decides: for class `DoS`, trust XGBoost 70% and Transformer 30%.

**Pipeline:** `prob_xgb × w_xgb + prob_tt × w_tt → argmax`  (20 trainable params)

In [ ]:
class WeightedVoting(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_w = nn.Parameter(torch.zeros(2, n_classes))

    def forward(self, prob_xgb, prob_tt):
        w = torch.softmax(self.log_w, dim=0)  # 2×10, columns sum to 1
        return w[0] * prob_xgb + w[1] * prob_tt

model_a3 = WeightedVoting().to(device)
print(f'Params: {sum(p.numel() for p in model_a3.parameters())} (just 20 weights)')

In [ ]:
print('\nTraining weighted soft voting...')

def get_tt_probs(X):
    tt_model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), 4096):
            batch = torch.from_numpy(X[i:i+4096]).to(device)
            _, out = tt_model(batch, return_embed=True)
            probs.append(torch.softmax(out, dim=1).cpu().numpy())
    return np.concatenate(probs, axis=0).astype(np.float32)

print('Getting TT probs...')
train_tt_prob = get_tt_probs(X_train_scale)
val_tt_prob   = get_tt_probs(X_val_scale)
test_tt_prob  = get_tt_probs(X_test_scale)

bs = 8192
train_loader = DataLoader(TensorDataset(
    torch.from_numpy(train_xgb_prob), torch.from_numpy(train_tt_prob), torch.from_numpy(y_train)
), batch_size=bs, shuffle=True)

criterion = nn.CrossEntropyLoss()
opt = optim.AdamW(model_a3.parameters(), lr=0.1)

for epoch in range(100):
    model_a3.train(); tr_loss = 0.0
    for xgb_p, tt_p, yb in train_loader:
        xgb_p, tt_p, yb = xgb_p.to(device), tt_p.to(device), yb.to(device)
        opt.zero_grad()
        pred = model_a3(xgb_p, tt_p)
        loss = criterion(torch.log(pred + 1e-8), yb)
        loss.backward(); opt.step()
        tr_loss += loss.item() * xgb_p.size(0)
    tr_loss /= len(y_train)
    if epoch % 20 == 0 or epoch == 0:
        print(f'Epoch {epoch:3d} | Loss: {tr_loss:.4f}')

# Show learned weights
with torch.no_grad():
    w = torch.softmax(model_a3.log_w, dim=0).cpu().numpy()
print('\nLearned weights (XGBoost vs Transformer per class):')
print(f"{'Class':<15} {'XGB':>6} {'TT':>6}")
print('-' * 27)
for i, name in enumerate(class_names):
    print(f'{name:<15} {w[0,i]:>6.3f} {w[1,i]:>6.3f}')

# Evaluate
model_a3.eval()
with torch.no_grad():
    preds = torch.argmax(model_a3(
        torch.from_numpy(test_xgb_prob),
        torch.from_numpy(test_tt_prob)
    ), dim=1).numpy()
macro_a3 = f1_score(y_test, preds, average='macro')
wgt_a3   = f1_score(y_test, preds, average='weighted')
print(f'\nApproach 3 (Weighted voting) → Macro F1: {macro_a3:.4f}, Weighted: {wgt_a3:.4f}')
print(classification_report(y_test, preds, target_names=class_names, digits=4))

In [ ]:
print('\n' + '=' * 60)
print('FINAL COMPARISON')
print('=' * 60)
print(f"{'Model':<40} {'Macro F1':>10} {'Weighted':>10}")
print('-' * 62)
print(f"{'XGBoost (baseline)':<40} {'0.6014':>10} {'0.9313':>10}")
print(f"{'Original Hybrid (logits + TT)':<40} {'0.5786':>10} {'0.9275':>10}")
print('-' * 62)
print(f"{'1. Leaf PCA + TT → Fusion MLP':<40} {macro_a1:>10.4f} {wgt_a1:>10.4f}")
print(f"{'2. Joint training (unfreeze TT)':<40} {macro_a2:>10.4f} {wgt_a2:>10.4f}")
print(f"{'3. Weighted soft voting (20 params)':<40} {macro_a3:>10.4f} {wgt_a3:>10.4f}")